In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
#from lazypredict.Supervised import LazyClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

df = pd.read_csv("term-deposit-marketing-2020.csv")

df["y"] = df["y"].map({"no": 0, "yes": 1})

binary_cols = ["housing", "loan", "default"]  # example

for col in binary_cols:
    df[col] = df[col].map({"no": 0, "yes": 1})

df = df.drop(columns="contact")

In [3]:
df.replace('unknown', np.nan, inplace=True)

,age,job,marital,education,default,balance,housing,loan,day,month,duration,campaign,y
0,58,management,married,tertiary,0,2143,1,0,5,may,261,1,0
1,44,technician,single,secondary,0,29,1,0,5,may,151,1,0
2,33,entrepreneur,married,secondary,0,2,1,1,5,may,76,1,0
3,47,blue-collar,married,NaN,0,1506,1,0,5,may,92,1,0
4,33,NaN,single,NaN,0,1,0,0,5,may,198,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,53,technician,married,tertiary,0,395,0,0,3,jun,107,1,0
39996,30,management,single,tertiary,0,3340,0,0,3,jun,238,3,1
39997,54,admin,divorced,secondary,0,200,0,0,3,jun,170,1,1
39998,34,management,married,tertiary,0,1047,0,0,3,jun,342,1,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        40000 non-null  int64
 1   job        39765 non-null  str  
 2   marital    40000 non-null  str  
 3   education  38469 non-null  str  
 4   default    40000 non-null  int64
 5   balance    40000 non-null  int64
 6   housing    40000 non-null  int64
 7   loan       40000 non-null  int64
 8   day        40000 non-null  int64
 9   month      40000 non-null  str  
 10  duration   40000 non-null  int64
 11  campaign   40000 non-null  int64
 12  y          40000 non-null  int64
dtypes: int64(9), str(4)
memory usage: 4.0 MB


In [5]:
known_education = df[df["education"].notna()].copy()
unknown_education = df[df["education"].isna()].copy()

In [6]:
education_features = ['age', 'job', 'marital', 'default', 'balance', 'housing', 'loan']
X_education = known_education[education_features]
#X_education = known_education.drop(["education", "y", "month"], axis=1)
y_education = known_education["education"]
X_missing = unknown_education[education_features]

In [7]:
X_education = pd.get_dummies(
    X_education,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [8]:
X_missing = pd.get_dummies(
    X_missing,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_education,
    y_education,
    test_size=0.2,
    random_state=1234,
    stratify=y_education
)

In [10]:
education_model = DecisionTreeClassifier(
    random_state=1234,
    max_depth=10,
    min_samples_leaf=5,
    class_weight={'primary': 2, 'secondary': 1, 'tertiary': 1}
)

In [11]:
education_model.fit(X_train, y_train)
education_test_pred = education_model.predict(X_test)

In [12]:
print(accuracy_score(y_test, education_test_pred))

0.6757213413049129


In [13]:
y_education.value_counts(normalize=True)

education
secondary    0.545712
tertiary     0.291299
primary      0.162988
Name: proportion, dtype: float64

In [14]:
print(confusion_matrix(y_test, education_test_pred))

[[ 816  371   67]
 [ 961 2961  277]
 [ 138  681 1422]]


In [15]:
print(education_model.classes_)

['primary' 'secondary' 'tertiary']


In [16]:
education_predictions = education_model.predict(X_missing)

df.loc[unknown_education.index, "education"] = education_predictions

In [17]:
df["education"].isna().sum()

np.int64(0)

In [18]:
df["education"] = df["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})

In [19]:
df = pd.get_dummies(
    df,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   age                40000 non-null  int64
 1   education          40000 non-null  int64
 2   default            40000 non-null  int64
 3   balance            40000 non-null  int64
 4   housing            40000 non-null  int64
 5   loan               40000 non-null  int64
 6   day                40000 non-null  int64
 7   month              40000 non-null  str  
 8   duration           40000 non-null  int64
 9   campaign           40000 non-null  int64
 10  y                  40000 non-null  int64
 11  job_blue-collar    40000 non-null  int64
 12  job_entrepreneur   40000 non-null  int64
 13  job_housemaid      40000 non-null  int64
 14  job_management     40000 non-null  int64
 15  job_retired        40000 non-null  int64
 16  job_self-employed  40000 non-null  int64
 17  job_services       4000